In [1]:
from dotenv import load_dotenv
import os
from langchain_community.graphs import Neo4jGraph

In [2]:
load_dotenv()
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

In [3]:
kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

/var/folders/_w/vvltlyns7qb080p3nh3nf0_w0000gn/T/ipykernel_4304/384792529.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)


# Modeling 

The entities are: 
- *Domain*
- *Topic*
- *Article*

Whereas the relationships are:
- *Domain* --[ENCOMPASSES]--> *Topic*
- *Topic* --[COVERS]--> *Article*
- *FirstArticle* --[RELATED_TO]--> *NextArticle*

In [4]:
## delete all nodes and relationships
kg.query("""
  MATCH (n)
  DETACH DELETE n
""")

[]

## Laws

In [5]:
# Create three law articles with descriptions
merge_article_node_query = """
MERGE (mergedArticle:Article {articleId: $articleParam.articleId})
    ON CREATE SET
        mergedArticle.title = $articleParam.title,
        mergedArticle.description = $articleParam.description,
        mergedArticle.topic = $articleParam.topic,
        mergedArticle.source = $articleParam.source
RETURN mergedArticle
"""

articles = [
    {
        "articleId": "art-001",
        "title": "Article 1 - Right to Life",
        "description": "Every person has the inherent right to life...",
        "topic": "Human Rights",
        "source": "European Convention on Human Rights"
    },
    {
        "articleId": "art-002",
        "title": "Article 2 - Freedom of Expression",
        "description": "Everyone has the right to freedom of expression...",
        "topic": "Human Rights",
        "source": "European Convention on Human Rights"
    },
    {
        "articleId": "art-003",
        "title": "Article 3 - Right to Fair Trial",
        "description": "Everyone is entitled to a fair and public hearing...",
        "topic": "Criminal Justice",
        "source": "European Convention on Human Rights"
    },
]

for article in articles:
    kg.query(merge_article_node_query, params={"articleParam": article})

## Topics

In [6]:
merge_topic_node_query = """
MERGE (mergedTopic:Topic {topicId: $topicParam.topicId})
    ON CREATE SET
        mergedTopic.name = $topicParam.name,
        mergedTopic.description = $topicParam.description,
        mergedTopic.domain = $topicParam.domain
RETURN mergedTopic
"""

topics = [
    {
        "topicId": "top-001",
        "name": "Human Rights",
        "description": "Fundamental rights and freedoms to which all humans are entitled",
        "domain": "Fundamental Law"
    },
    {
        "topicId": "top-002",
        "name": "Criminal Justice",
        "description": "System of practices and institutions directed at upholding social control and deterring crime",
        "domain": "Fundamental Law"
    },
]

for topic in topics:
    kg.query(merge_topic_node_query, params={"topicParam": topic})

print("Topics merged.")

Topics merged.


## Domain

In [7]:
merge_domain_node_query = """
MERGE (mergedDomain:Domain {domainId: $domainParam.domainId})
    ON CREATE SET
        mergedDomain.name = $domainParam.name,
        mergedDomain.description = $domainParam.description
RETURN mergedDomain
"""

kg.query(merge_domain_node_query, params={"domainParam": {
    "domainId": "dom-001",
    "name": "Fundamental Law",
    "description": "The body of law that governs the basic rights and freedoms of individuals"
}})

print("Domain merged.")

Domain merged.


## Create relationships

In [8]:
merge_relationships_query = """
MATCH (d:Domain {domainId: $domainId})
MATCH (t:Topic {topicId: $topicId})
MERGE (d)-[:ENCOMPASSES]->(t)
"""

kg.query(merge_relationships_query, params={"domainId": "dom-001", "topicId": "top-001"})
kg.query(merge_relationships_query, params={"domainId": "dom-001", "topicId": "top-002"})

merge_topic_article_query = """
MATCH (t:Topic {topicId: $topicId})
MATCH (a:Article {articleId: $articleId})
MERGE (t)-[:COVERS]->(a)
"""

kg.query(merge_topic_article_query, params={"topicId": "top-001", "articleId": "art-001"})
kg.query(merge_topic_article_query, params={"topicId": "top-001", "articleId": "art-002"})
kg.query(merge_topic_article_query, params={"topicId": "top-002", "articleId": "art-003"})


[]

In [9]:
print("Relationships created. Verifying...")
results = kg.query("""
  MATCH (d:Domain)-[:ENCOMPASSES]->(t:Topic)-[:COVERS]->(a:Article)
  RETURN d.name AS domain, t.name AS topic, a.title AS article
  ORDER BY t.name
""")
for r in results:
    print(f"{r['domain']}  →  {r['topic']}  →  {r['article']}")

Relationships created. Verifying...
Fundamental Law  →  Criminal Justice  →  Article 3 - Right to Fair Trial
Fundamental Law  →  Human Rights  →  Article 1 - Right to Life
Fundamental Law  →  Human Rights  →  Article 2 - Freedom of Expression


In [10]:
merge_articles_same_topic_query = """
MATCH (from_same_topic:Article)
  WHERE from_same_topic.topic = $topicParam
  WITH from_same_topic
    ORDER BY from_same_topic.articleId ASC
  WITH collect(from_same_topic) as topic_article_list
    CALL apoc.nodes.link(
        topic_article_list, 
        "RELATED_TO", 
        {avoidDuplicates: true}
    )
RETURN size(topic_article_list)
"""

kg.query(merge_articles_same_topic_query, params={"topicParam": "Human Rights"})

[{'size(topic_article_list)': 2}]

In [11]:
section_relationships_first_article_query = """
   MATCH (t:Topic)-[:COVERS]->(a:Article)
   WHERE t.name = $topicName
   ORDER BY a.articleId ASC LIMIT 1
   MERGE (t)-[:FIRST_ARTICLE_DOMAIN]->(a)
"""

list_topics = ["Human Rights", "Criminal Justice"]
for topic in list_topics:
   kg.query(section_relationships_first_article_query, params={"topicName": topic})

## Count total number of nodes

In [12]:
kg.query("""
         MATCH (n)
         RETURN count(n) as nodeCount
         """)

[{'nodeCount': 6}]

In [13]:
kg.refresh_schema()
print(kg.get_schema)

Node properties:
Article {source: STRING, topic: STRING, title: STRING, articleId: STRING, description: STRING}
Topic {topicId: STRING, domain: STRING, name: STRING, description: STRING}
Domain {domainId: STRING, name: STRING, description: STRING}
Relationship properties:

The relationships:
(:Article)-[:RELATED_TO]->(:Article)
(:Topic)-[:FIRST_ARTICLE_DOMAIN]->(:Article)
(:Topic)-[:COVERS]->(:Article)
(:Domain)-[:ENCOMPASSES]->(:Topic)


## Create a Vector Index

References
* [huggingface-notebook-example](https://colab.research.google.com/github/huggingface/cookbook/blob/main/notebooks/en/rag_with_knowledge_graphs_neo4j.ipynb#scrollTo=QNNoWuB3A8hZ)